# Neural Network Architectures in PyTorch

Hands-on implementation of all major neural network architectures — from MLPs through transformers. Each section builds from scratch using `nn.Module`, then validates against PyTorch built-ins and analyses the key properties (gradient flow, parameter efficiency, attention patterns).

| Section | Topic | Key experiment |
|---------|-------|----------------|
| 1 | MLP: depth and non-linearity | SVD rank collapse without activations |
| 2 | CNN and depthwise separable | Output-size formula; parameter counts |
| 3 | ResNet residual connections | Gradient magnitude vs. depth |
| 4 | Vanilla RNN vs. LSTM | Long-sequence accuracy and gradient norms |
| 5 | Scaled dot-product attention | Softmax saturation without √d_k scaling |
| 6 | Mini decoder-only transformer | Character LM; attention pattern visualisation |

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')

## Section 1 — MLP: Depth, Width, and Non-linearity

We build a configurable MLP and verify two things:
1. **Parameter count** matches the theoretical formula.
2. **Rank collapse**: a deep linear MLP (no activations) has output rank ≤ `d_in`, regardless of hidden width — it is equivalent to a single matrix multiplication.

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_in, hidden_dims, d_out, activation=nn.ReLU):
        super().__init__()
        dims = [d_in] + list(hidden_dims) + [d_out]
        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:   # no activation after the final linear
                layers.append(activation())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def expected_params(d_in, hidden_dims, d_out):
    """Weights + biases for each layer."""
    dims = [d_in] + list(hidden_dims) + [d_out]
    total = 0
    for i in range(len(dims) - 1):
        total += dims[i] * dims[i + 1] + dims[i + 1]   # weight + bias
    return total


# --- Verify parameter counts ---
configs = [
    (16, [64, 64], 10),
    (784, [256, 128, 64], 10),
    (32, [512], 1),
]
print('Parameter count verification:')
for d_in, hidden, d_out in configs:
    m = MLP(d_in, hidden, d_out)
    actual   = count_params(m)
    expected = expected_params(d_in, hidden, d_out)
    ok = '✓' if actual == expected else '✗'
    print(f'  {ok} MLP({d_in}, {hidden}, {d_out}): actual={actual}  expected={expected}')

In [ ]:
# --- Rank collapse experiment ---
# A composition of linear maps is still a linear map.
# Output rank ≤ min(d_in, hidden_dims...) regardless of depth.
torch.manual_seed(0)
d_in, d_out = 8, 32
hidden = [64, 64, 64]   # wide hidden layers

mlp_linear = MLP(d_in, hidden, d_out, activation=nn.Identity)
mlp_relu   = MLP(d_in, hidden, d_out, activation=nn.ReLU)

x = torch.randn(256, d_in)
with torch.no_grad():
    out_linear = mlp_linear(x)   # (256, 32)
    out_relu   = mlp_relu(x)

rank_linear = torch.linalg.matrix_rank(out_linear).item()
rank_relu   = torch.linalg.matrix_rank(out_relu).item()

print(f'Linear MLP (no activations): output rank = {rank_linear}  (max possible = {d_in})')
print(f'ReLU MLP:                    output rank = {rank_relu}   (max possible = {min(256, d_out)})')
assert rank_linear <= d_in, 'Expected rank ≤ d_in for linear MLP'
print(f'\n✓ Linear MLP rank ({rank_linear}) ≤ d_in ({d_in}) — collapse confirmed')

## Section 2 — CNN and Depthwise Separable Convolutions

A standard `Conv2d(in_ch, out_ch, K)` costs `in_ch × out_ch × K²` parameters (plus bias).
A depthwise separable convolution factorises this into:
- **Depthwise**: `in_ch × K²` — one filter per input channel
- **Pointwise**: `in_ch × out_ch` — 1×1 conv to mix channels

Total ≈ `in_ch × (K² + out_ch)`, giving roughly a `out_ch / (1 + out_ch/K²)` × reduction.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, K=3, S=1, P=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, K, stride=S, padding=P, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)))


class DepthwiseSeparable(nn.Module):
    def __init__(self, in_ch, out_ch, K=3):
        super().__init__()
        P = K // 2
        self.dw = nn.Conv2d(in_ch, in_ch,  K, padding=P, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        return F.relu(self.bn(self.pw(self.dw(x))))


# --- Output-size formula verification ---
# H_out = floor((H_in + 2P - K) / S) + 1
print('Output-size formula verification (H_in=32):')
for K, S, P in [(3, 1, 1), (3, 2, 1), (5, 1, 2), (1, 1, 0)]:
    H_in = 32
    expected = math.floor((H_in + 2 * P - K) / S) + 1
    x = torch.randn(1, 16, H_in, H_in)
    out = nn.Conv2d(16, 32, K, stride=S, padding=P)(x)
    actual = out.shape[2]
    ok = '✓' if actual == expected else '✗'
    print(f'  {ok} K={K} S={S} P={P}: expected={expected}  actual={actual}')

# --- Parameter count comparison ---
in_ch, out_ch, K = 64, 128, 3
std_params = in_ch * out_ch * K * K
dws_params = in_ch * K * K + in_ch * out_ch   # dw + pw
ratio = std_params / dws_params
print(f'\nParameter reduction (in={in_ch}, out={out_ch}, K={K}):')
print(f'  Standard conv:        {std_params:,} params')
print(f'  Depthwise separable:  {dws_params:,} params')
print(f'  Reduction factor:     {ratio:.1f}×')

In [ ]:
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

transform = T.Compose([T.ToTensor(), T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
train_set = torchvision.datasets.CIFAR10('/tmp/cifar10', train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR10('/tmp/cifar10', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)
print(f'CIFAR-10: {len(train_set)} train / {len(test_set)} test')

In [ ]:
class CNN5Block(nn.Module):
    def __init__(self, use_dws=False):
        super().__init__()
        block = DepthwiseSeparable if use_dws else ConvBlock
        self.features = nn.Sequential(
            ConvBlock(3, 32),           # always standard for first layer (3 → 32)
            nn.MaxPool2d(2),
            block(32, 64),
            nn.MaxPool2d(2),
            block(64, 128),
            nn.MaxPool2d(2),
            block(128, 128),
            block(128, 256),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def train_cnn(model, n_epochs=5):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    train_losses, val_accs = [], []
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(imgs), labels)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        scheduler.step()
        train_losses.append(total_loss / len(train_loader))
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                correct += (model(imgs).argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_accs.append(correct / total)
        print(f'  Epoch {epoch+1}: loss={train_losses[-1]:.3f}  val_acc={val_accs[-1]:.3f}')
    return train_losses, val_accs


torch.manual_seed(1)
cnn_std = CNN5Block(use_dws=False)
cnn_dws = CNN5Block(use_dws=True)
print(f'Standard CNN params: {count_params(cnn_std):,}')
print(f'DWS CNN params:      {count_params(cnn_dws):,}')
print(f'Reduction: {count_params(cnn_std)/count_params(cnn_dws):.1f}×')

print('\nTraining standard CNN:')
losses_std, accs_std = train_cnn(cnn_std)
print('\nTraining depthwise-separable CNN:')
losses_dws, accs_dws = train_cnn(cnn_dws)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(losses_std, 'o-', label='Standard CNN')
axes[0].plot(losses_dws, 's-', label='DWS CNN')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(accs_std, 'o-', label='Standard CNN')
axes[1].plot(accs_dws, 's-', label='DWS CNN')
axes[1].set_title('Val Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Section 3 — ResNet Residual Connections and Gradient Flow

In a plain deep network, gradients in early layers diminish (or explode) with depth. Residual connections create a shortcut path `x → x + F(x)` whose gradient is `1 + ∂F/∂x`, keeping the gradient above 1 and preventing vanishing.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + identity)


class BottleneckBlock(nn.Module):
    expansion = 4

    def __init__(self, in_ch, bottleneck_ch, downsample=None):
        super().__init__()
        out_ch = bottleneck_ch * self.expansion
        self.conv1 = nn.Conv2d(in_ch,          bottleneck_ch, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(bottleneck_ch)
        self.conv2 = nn.Conv2d(bottleneck_ch, bottleneck_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(bottleneck_ch)
        self.conv3 = nn.Conv2d(bottleneck_ch, out_ch,         1, bias=False)
        self.bn3   = nn.BatchNorm2d(out_ch)
        self.downsample = downsample

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        return F.relu(out + identity)


def make_plain_net(n_blocks=10, channels=32):
    """Plain CNN: n_blocks BasicBlock-shaped layers, no residual."""
    layers = [nn.Conv2d(3, channels, 3, padding=1, bias=False), nn.BatchNorm2d(channels), nn.ReLU()]
    for _ in range(n_blocks):
        layers += [
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
        ]
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(channels, 10)]
    return nn.Sequential(*layers)


def make_resnet(n_blocks=10, channels=32):
    """ResNet: n_blocks BasicBlocks with residual connections."""
    layers = [nn.Conv2d(3, channels, 3, padding=1, bias=False), nn.BatchNorm2d(channels), nn.ReLU()]
    for _ in range(n_blocks):
        layers.append(BasicBlock(channels))
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(channels, 10)]
    return nn.Sequential(*layers)


print('Built plain CNN and ResNet (10 blocks each)')
print(f'  Plain:  {count_params(make_plain_net()):,} params')
print(f'  ResNet: {count_params(make_resnet()):,} params')

In [ ]:
# Gradient flow experiment
torch.manual_seed(2)
N_BLOCKS = 10

plain  = make_plain_net(N_BLOCKS).to(device)
resnet = make_resnet(N_BLOCKS).to(device)

x_batch = torch.randn(16, 3, 32, 32, device=device)
y_batch = torch.randint(0, 10, (16,), device=device)

def get_conv_grad_norms(model):
    """Run one forward+backward pass and return gradient norms for each Conv2d weight."""
    model.zero_grad()
    loss = F.cross_entropy(model(x_batch), y_batch)
    loss.backward()
    norms = []
    for m in model.modules():
        if isinstance(m, nn.Conv2d) and m.weight.grad is not None:
            norms.append(m.weight.grad.norm().item())
    return norms

norms_plain  = get_conv_grad_norms(plain)
norms_resnet = get_conv_grad_norms(resnet)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, norms, title in [
    (axes[0], [norms_plain, norms_resnet], 'linear'),
    (axes[1], [norms_plain, norms_resnet], 'log'),
]:
    fn = ax.plot if title == 'linear' else ax.semilogy
    fn(norms[0], 'o-', label='Plain CNN',  linewidth=2, markersize=5)
    fn(norms[1], 's-', label='ResNet',     linewidth=2, markersize=5)
    ax.set_xlabel('Conv layer index (shallow → deep)')
    ax.set_ylabel('Gradient norm' + (' (log)' if title == 'log' else ''))
    ax.set_title(f'Gradient Flow — {title} scale')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Plain  — first/last grad norm ratio: {norms_plain[0]/max(norms_plain[-1], 1e-30):.2f}')
print(f'ResNet — first/last grad norm ratio: {norms_resnet[0]/max(norms_resnet[-1], 1e-30):.2f}')

## Section 4 — Vanilla RNN vs. LSTM

Vanilla RNNs suffer from vanishing gradients on long sequences: the hidden state is updated by `h_t = tanh(W_hh h_{t-1} + W_xh x_t)`, and gradients are multiplied by `W_hh^T` at each timestep. An LSTM's cell-state path has an additive update, keeping gradients larger over long sequences.

In [ ]:
class VanillaRNN(nn.Module):
    def __init__(self, d_in, d_h, d_out):
        super().__init__()
        self.d_h = d_h
        self.Wh  = nn.Linear(d_h,  d_h, bias=False)
        self.Wx  = nn.Linear(d_in, d_h)
        self.out = nn.Linear(d_h,  d_out)

    def forward(self, x):     # x: (B, T, d_in)
        h = torch.zeros(x.size(0), self.d_h, device=x.device)
        for t in range(x.size(1)):
            h = torch.tanh(self.Wh(h) + self.Wx(x[:, t]))
        return self.out(h)


class LSTMModel(nn.Module):
    def __init__(self, d_in, d_h, d_out):
        super().__init__()
        self.d_h  = d_h
        # Gates: input, forget, cell, output — concatenated
        self.Wih  = nn.Linear(d_in, 4 * d_h)
        self.Whh  = nn.Linear(d_h,  4 * d_h, bias=False)
        self.out  = nn.Linear(d_h,  d_out)

    def forward(self, x):     # x: (B, T, d_in)
        B = x.size(0)
        h = torch.zeros(B, self.d_h, device=x.device)
        c = torch.zeros(B, self.d_h, device=x.device)
        for t in range(x.size(1)):
            gates = self.Wih(x[:, t]) + self.Whh(h)
            i, f, g, o = gates.chunk(4, dim=1)
            i = torch.sigmoid(i)
            f = torch.sigmoid(f)
            g = torch.tanh(g)
            o = torch.sigmoid(o)
            c = f * c + i * g
            h = o * torch.tanh(c)
        return self.out(h)


# Long-sequence synthetic task:
# Label = sign of x[0, 0]. Signal is buried at position 0; sequence length = 200.
def make_long_seq_dataset(n=2000, seq_len=200, d_in=8):
    torch.manual_seed(3)
    X = torch.randn(n, seq_len, d_in)
    y = (X[:, 0, 0] > 0).long()   # label from first timestep only
    return X, y


X, y = make_long_seq_dataset()
split = 1600
X_train, y_train = X[:split], y[:split]
X_val,   y_val   = X[split:], y[split:]
print(f'Dataset: {X_train.shape} train, {X_val.shape} val  (seq_len=200, signal at t=0)')

In [ ]:
def train_rnn(model, n_epochs=10, lr=1e-3, batch_size=64):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    X_tr, y_tr = X_train.to(device), y_train.to(device)
    X_v,  y_v  = X_val.to(device),   y_val.to(device)
    accs = []
    for epoch in range(n_epochs):
        model.train()
        idx = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), batch_size):
            b = idx[i:i+batch_size]
            loss = F.cross_entropy(model(X_tr[b]), y_tr[b])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            preds = model(X_v).argmax(1)
            acc = (preds == y_v).float().mean().item()
        accs.append(acc)
        print(f'  Epoch {epoch+1}: val_acc={acc:.3f}')
    return accs


torch.manual_seed(4)
print('Vanilla RNN:')
rnn_model  = VanillaRNN(8, 64, 2)
accs_rnn   = train_rnn(rnn_model)

torch.manual_seed(4)
print('\nLSTM:')
lstm_model = LSTMModel(8, 64, 2)
accs_lstm  = train_rnn(lstm_model)

plt.figure(figsize=(8, 4))
plt.plot(accs_rnn,  'o-', label='Vanilla RNN', linewidth=2)
plt.plot(accs_lstm, 's-', label='LSTM',        linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Val accuracy')
plt.title('Long-Sequence Task (seq_len=200, signal at t=0)')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='chance')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Gradient norms at first timestep
def first_timestep_grad_norm(model, seq_len=200, d_in=8):
    model = model.to(device)
    model.zero_grad()
    x = torch.randn(32, seq_len, d_in, device=device, requires_grad=False)
    # Run through all timesteps manually, keeping the first hidden state
    # We want the gradient of the loss w.r.t. the input at t=0
    x_req = x.clone().detach().requires_grad_(True)
    out = model(x_req)
    loss = F.cross_entropy(out, torch.randint(0, 2, (32,), device=device))
    loss.backward()
    # Gradient w.r.t. x at first timestep
    return x_req.grad[:, 0, :].norm().item()

norm_rnn  = first_timestep_grad_norm(rnn_model)
norm_lstm = first_timestep_grad_norm(lstm_model)
print(f'Gradient norm at t=0:')
print(f'  Vanilla RNN: {norm_rnn:.4e}')
print(f'  LSTM:        {norm_lstm:.4e}')
print(f'  LSTM/RNN ratio: {norm_lstm/max(norm_rnn, 1e-30):.1f}×  (LSTM should be larger)')

## Section 5 — Scaled Dot-Product Attention

Attention computes `softmax(QK^T / √d_k) V`. The `√d_k` scaling prevents the dot products from growing large as `d_k` increases — without it, the softmax saturates (one weight approaches 1, all others ≈ 0) and gradients vanish.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None, scale=True):
    """Q,K,V: (..., seq, d_k). Returns (output, weights)."""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1)   # (..., seq, seq)
    if scale:
        scores = scores / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return weights @ V, weights


# Saturation experiment: vary d_k, measure entropy of attention weights
def softmax_entropy(weights):
    """Shannon entropy of attention weight distribution (nats)."""
    eps = 1e-9
    return -(weights * (weights + eps).log()).sum(-1).mean().item()


d_k_values = [4, 8, 16, 32, 64, 128, 256, 512]
entropies_scaled   = []
entropies_unscaled = []
torch.manual_seed(5)
for d_k in d_k_values:
    Q = torch.randn(64, 16, d_k)   # (batch, seq, d_k)
    K = torch.randn(64, 16, d_k)
    V = torch.randn(64, 16, d_k)
    _, w_scaled   = scaled_dot_product_attention(Q, K, V, scale=True)
    _, w_unscaled = scaled_dot_product_attention(Q, K, V, scale=False)
    entropies_scaled.append(softmax_entropy(w_scaled))
    entropies_unscaled.append(softmax_entropy(w_unscaled))

plt.figure(figsize=(8, 4))
plt.plot(d_k_values, entropies_scaled,   'o-', label='With √d_k scaling',    linewidth=2)
plt.plot(d_k_values, entropies_unscaled, 's-', label='Without √d_k scaling', linewidth=2)
plt.xlabel('d_k'); plt.ylabel('Softmax entropy (nats)')
plt.title('Attention Weight Entropy vs d_k')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print('Entropy at d_k=512:')
print(f'  Scaled:   {entropies_scaled[-1]:.4f}  nats')
print(f'  Unscaled: {entropies_unscaled[-1]:.4f}  nats  (should be much lower — saturated)')

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads
        self.W_q     = nn.Linear(d_model, d_model, bias=False)
        self.W_k     = nn.Linear(d_model, d_model, bias=False)
        self.W_v     = nn.Linear(d_model, d_model, bias=False)
        self.W_o     = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, causal_mask=False):
        B, T, D = x.shape
        H, dk   = self.n_heads, self.d_k
        Q = self.W_q(x).view(B, T, H, dk).transpose(1, 2)  # (B, H, T, dk)
        K = self.W_k(x).view(B, T, H, dk).transpose(1, 2)
        V = self.W_v(x).view(B, T, H, dk).transpose(1, 2)
        mask = None
        if causal_mask:
            mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        out, self.last_weights = scaled_dot_product_attention(Q, K, V, mask=mask)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)


# Verify output shape
torch.manual_seed(6)
mha = MultiHeadAttention(d_model=128, n_heads=8)
x_test = torch.randn(4, 10, 128)
out = mha(x_test)
print(f'MHA output shape: {tuple(out.shape)}  (expected: (4, 10, 128))')
assert out.shape == (4, 10, 128)
print('✓ Shape correct')

# Verify against nn.MultiheadAttention
ref = nn.MultiheadAttention(128, 8, bias=False, batch_first=True)
out_ref, _ = ref(x_test, x_test, x_test, need_weights=False)
print(f'Output std (ours): {out.std().item():.4f}  |  ref: {out_ref.std().item():.4f}  (similar order of magnitude)')

## Section 6 — Mini Decoder-Only Transformer

We assemble a character-level language model using the `MultiHeadAttention` from Section 5. The model is a stack of `TransformerBlock`s with pre-layer-norm, a feed-forward network (FFN) with GELU, and dropout. We train it on a fixed text string for a few hundred steps and visualise per-head attention patterns.

In [ ]:
def sinusoidal_pe(T, d_model):
    """Sinusoidal positional encoding: PE[pos, 2i] = sin(pos/10000^(2i/d_model))."""
    pe  = torch.zeros(T, d_model)
    pos = torch.arange(T).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div[:d_model // 2])
    return pe   # (T, d_model)


# Verify shift-by-dot-product property: PE(pos+k) ≈ R(k) @ PE(pos)
d = 64; T = 100
pe = sinusoidal_pe(T, d)
# The dot product PE(pos) · PE(pos+k) depends only on k, not pos
dots = [pe[20].dot(pe[20 + k]).item() for k in range(10)]
dots2 = [pe[50].dot(pe[50 + k]).item() for k in range(10)]
print('PE dot products — should depend on offset k, not on absolute position:')
print('  pos=20:', [f'{v:.2f}' for v in dots])
print('  pos=50:', [f'{v:.2f}' for v in dots2])

# Visualise PE
plt.figure(figsize=(10, 3))
plt.imshow(pe[:50, :64].numpy(), aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar()
plt.title('Sinusoidal Positional Encoding (first 50 positions, 64 dims)')
plt.xlabel('Dimension'); plt.ylabel('Position'); plt.tight_layout(); plt.show()

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = MultiHeadAttention(d_model, n_heads)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, causal_mask=True):
        x = x + self.drop(self.attn(self.norm1(x), causal_mask=causal_mask))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=3, d_ff=256,
                 max_len=128, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.register_buffer('pe', sinusoidal_pe(max_len, d_model))
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.norm  = nn.LayerNorm(d_model)
        self.head  = nn.Linear(d_model, vocab_size, bias=False)
        self.drop  = nn.Dropout(dropout)

    def forward(self, idx):   # idx: (B, T)
        T   = idx.size(1)
        x   = self.drop(self.embed(idx) + self.pe[:T])
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x))


# Build character-level dataset from a fixed text string
TEXT = (
    "To be, or not to be, that is the question: whether 'tis nobler in the mind to suffer "
    "the slings and arrows of outrageous fortune, or to take arms against a sea of troubles. "
    "All the world's a stage, and all the men and women merely players. "
    "Friends, Romans, countrymen, lend me your ears. I come to bury Caesar, not to praise him. "
    "What light through yonder window breaks? It is the east, and Juliet is the sun. "
) * 10   # repeat to give enough training data

chars    = sorted(set(TEXT))
stoi     = {ch: i for i, ch in enumerate(chars)}
itos     = {i: ch for ch, i in stoi.items()}
vocab_sz = len(chars)
data     = torch.tensor([stoi[c] for c in TEXT], dtype=torch.long)

CTX = 64   # context length
print(f'Vocab size: {vocab_sz}  |  Dataset length: {len(data)} chars  |  Context: {CTX}')

In [ ]:
def get_batch(data, batch_size=32, ctx=CTX):
    idx = torch.randint(len(data) - ctx, (batch_size,))
    x   = torch.stack([data[i:i+ctx]   for i in idx])
    y   = torch.stack([data[i+1:i+ctx+1] for i in idx])
    return x.to(device), y.to(device)


torch.manual_seed(7)
model_gpt = MiniGPT(vocab_sz, d_model=128, n_heads=4, n_layers=3, d_ff=256, max_len=CTX).to(device)
opt = torch.optim.AdamW(model_gpt.parameters(), lr=3e-3)

losses_gpt = []
N_STEPS = 400
for step in range(N_STEPS):
    xb, yb = get_batch(data)
    logits  = model_gpt(xb)          # (B, T, vocab_sz)
    loss    = F.cross_entropy(logits.view(-1, vocab_sz), yb.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    losses_gpt.append(loss.item())
    if (step + 1) % 100 == 0:
        print(f'  Step {step+1}/{N_STEPS}: loss={loss.item():.4f}')

plt.figure(figsize=(8, 3))
w = 20
smoothed = np.convolve(losses_gpt, np.ones(w)/w, mode='valid')
plt.plot(losses_gpt, alpha=0.3, color='steelblue')
plt.plot(range(w-1, len(losses_gpt)), smoothed, color='steelblue', linewidth=2, label='smoothed')
plt.xlabel('Step'); plt.ylabel('Cross-entropy loss')
plt.title('Mini Transformer Training Loss'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Visualise per-head attention patterns in the last block
model_gpt.eval()
with torch.no_grad():
    sample_idx = data[:CTX].unsqueeze(0).to(device)
    _ = model_gpt(sample_idx)
    attn_weights = model_gpt.blocks[-1].attn.last_weights   # (1, n_heads, T, T)

n_heads = attn_weights.size(1)
fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))
sample_text = TEXT[:CTX]
tick_labels = list(sample_text[::8])   # every 8th char
tick_pos    = list(range(0, CTX, 8))

for h, ax in enumerate(axes):
    w = attn_weights[0, h].cpu().numpy()   # (T, T)
    im = ax.imshow(w, cmap='viridis', aspect='auto', vmin=0)
    ax.set_title(f'Head {h+1}', fontsize=11)
    ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels, fontsize=7)
    ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Per-Head Attention Patterns (final block)', y=1.02)
plt.tight_layout(); plt.show()
print('Different heads should show qualitatively different patterns (local, global, positional).')